# Importing Necessary Libs

In [ ]:
import os
import re
import gc
import json
import time
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import matplotlib.dates as mdates
import seaborn as sns

import rasterio
from rasterio.enums import Resampling
from scipy import stats as scipy_stats
from scipy.signal import savgol_filter
from pyproj import Transformer

import joblib

from sklearn.ensemble import RandomForestClassifier,GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (accuracy_score, classification_report,confusion_matrix, r2_score, mean_squared_error,cohen_kappa_score)

# Only keeping the real errors visible, and ignoring the warnings that are not relevant to the analysis
warnings.filterwarnings("ignore", category=UserWarning,  module="rasterio")
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*invalid value encountered.*")
warnings.filterwarnings("ignore", message=".*divide by zero.*")


In [ ]:
plt.rcParams.update({
    "font.family":       "DejaVu Sans",
    "font.size":         11,
    "axes.titlesize":    13,
    "figure.dpi":        110,
    "figure.facecolor":  "white",
    "axes.facecolor":    "white",
    "axes.spines.right": False,
    "axes.spines.top":   False,
})

# Defining Constants

In [ ]:
RAW_DIR    = "data"
OUT_DIR    = "out"
DOWNSAMPLE = 2 #Taking the half resolution to speedup


# Scaling and offset for converting raw digital numbers to surface reflectance, based on Landsat 8 and 9 images
# Surface Reflectance = Raw Digital Number * 0.0000275 + (−0.2)
SCALE  = 0.0000275
OFFSET = -0.2

# Quality Assesment Layer Defining
QA_FILL = 0x0001
QA_DILCLD = 0x0002
QA_CIRRUS = 0x0004
QA_CLOUD = 0x0008
QA_SHADOW = 0x0010
QA_CLOUD_MASK = QA_FILL | QA_DILCLD | QA_CIRRUS | QA_CLOUD | QA_SHADOW

# Land cover classes
CLASS_NAMES = {0: "Water", 1: "Vegetation", 2: "Bare Soil", 3: "Built-Up"}
CLASS_COLORS = {-1: "gray", 0: "blue", 1: "#green", 2: "saddlebrown", 3: "#red"}

VEG_THRESHOLD = 0.25
DOWNSAMPLE = int(DOWNSAMPLE)
PX_HA = (30 * DOWNSAMPLE) ** 2 / 10_000 #Calculating the area of each pixel in hectares, based on the downsampled resolution
FEATURE_NAMES = ["B2","B3","B4","B5","B6","B7","NDVI","NDWI","NDBI","NBR","EVI"]
# B2, B3, B4: Blue, Green, and Red light
# B5: NearInfrared strongly reflected by healthy plants.
# B6, B7: Shortwave Infrared (SWIR 1 and 2) great for spotting rock, soil, and moisture.

# NDVI (Normalized Difference Vegetation Index): Highlights green plants.
# NDWI (Normalized Difference Water Index): Highlights open water.
# NDBI (Normalized Difference Built-Up Index): Highlights buildings, concrete, and asphalt.
# NBR (Normalized Burn Ratio): Highlights burnt land/dry soil.
# EVI (Enhanced Vegetation Index): A backup for NDVI that works better in heavy, dense jungles.


# Creating the directories for the output
for sub in ["stacked","ndvi","ndwi","ndbi","nbr","evi","classmaps","change","graphs","stats","models"]:
    os.makedirs(os.path.join(OUT_DIR, sub), exist_ok=True)



# Creating Helper Functions

### Finding and Reading files

In [ ]:
#Using regex to find the files identify name and extract key data like satelite name, acquisition date and spectral band
def parse_landsat_name(fname):
    base = os.path.basename(fname)
    m = re.match(
        r"(LC0[89])_L2SP_(\d{6})_(\d{8})_\d{8}_\d{2}_T\d_"
        r"(SR_B[1-7]|QA_PIXEL|QA_RADSAT|SR_QA_AEROSOL)",
        base, re.IGNORECASE
    )
    if not m:
        return None
    acq = m.group(3)
    return {
        "sensor": m.group(1).upper(),
        "path_row": m.group(2),
        "date": datetime.strptime(acq, "%Y%m%d"),
        "band": m.group(4).upper(),
        "label": f"{acq[:4]}_{acq[4:6]}",
    }


In [ ]:
#Scaning the raw dara folders to and filter unrelevant files, and organize them by date

def discover_scenes(raw_dir):
    raw_path = Path(raw_dir)
    if not raw_path.exists():
        raise FileNotFoundError(f"Raw data directory not found: {raw_dir}")
    scenes = {}
    for folder in sorted(raw_path.iterdir()):
        if not folder.is_dir():
            continue
        tifs = sorted(list(folder.glob("*.TIF")) + list(folder.glob("*tif")))
        if not tifs:
            print(f"  {folder.name}: no TIF files")
            continue
        entry = {}
        first_info = None
        for tif in tifs:
            info = parse_landsat_name(tif.name)
            if info:
                entry[info["band"]] = str(tif)
                if first_info is None:
                    first_info = info
        if not entry or first_info is None:
            print(f"  {folder.name}: no Landsat-named TIFs")
            continue
        entry["_meta"] = {
            "sensor": first_info["sensor"],
            "date": first_info["date"],
            "label": first_info["label"],
            "path_row": first_info["path_row"],
        }
        scenes[first_info["label"]] = entry
    return scenes

### Image Resizing Calibration and Masking

In [ ]:
#open the spectral image files, if downsample is set to 1, it will load the image at its original resolution, otherwise it will load a downsampled version of the image using bilinear resampling. Also handles empty pixels by setting them to NaN.

def load_band(path, downsample=1):
    with rasterio.open(path) as src:
        if downsample == 1:
            arr     = src.read(1).astype(np.float32)
            profile = src.profile.copy()
        else:
            out_h = src.height // downsample
            out_w = src.width  // downsample
            arr   = src.read(
                1, out_shape=(out_h, out_w),
                resampling=Resampling.bilinear
            ).astype(np.float32)
            new_t   = src.transform * src.transform.scale(
                src.width / out_w, src.height / out_h
            )
            profile = src.profile.copy()
            profile.update(height=out_h, width=out_w, transform=new_t)
        nodata = src.nodata
    if nodata is not None:
        arr[arr == nodata] = np.nan
    arr = np.clip(arr * SCALE + OFFSET, 0.0, 1.0)
    return arr, profile

In [ ]:
#reading the qa layer and if it scales down it uses nearest neighbour resampling by picking the closest single value, and returns a boolean mask where True indicates a bad pixel based on the defined QA_CLOUD_MASK.

def load_qa(path, downsample=1):
    with rasterio.open(path) as src:
        if downsample == 1:
            qa = src.read(1).astype(np.uint16)
        else:
            out_h = src.height // downsample
            out_w = src.width  // downsample
            qa = src.read(
                1, out_shape=(out_h, out_w),
                resampling=Resampling.nearest
            ).astype(np.uint16)
    return (qa & QA_CLOUD_MASK) != 0

### Image Post Processing

In [ ]:
# Exports the processd arrays back into geo referenced TIFF files and using LZW compression to reduce file size, and setting nodata values to NaN for better handling in future analysis.

def save_raster(arr, profile, out_path):
    os.makedirs(os.path.dirname(os.path.abspath(out_path)), exist_ok=True)
    p = profile.copy()
    p.update(count=1, dtype="float32", compress="lzw",
             nodata=float("nan"), BIGTIFF="YES")
    with rasterio.open(out_path, "w", **p) as dst:
        dst.write(arr[np.newaxis, :, :])

In [ ]:
#Mapping the pixel values between 0 and 1 based on the specified percentiles, this prevents outliers from skewing the contrast and allows for better visualization of the images

def pct_stretch(band, pct_lo=2, pct_hi=98):
    valid = band[~np.isnan(band)]
    if len(valid) == 0:
        return np.zeros_like(band, dtype=np.float32)
    lo, hi = np.percentile(valid, [pct_lo, pct_hi])
    if hi == lo:
        return np.zeros_like(band, dtype=np.float32)
    return np.clip((band - lo) / (hi - lo), 0.0, 1.0).astype(np.float32)

In [ ]:
#Instead of reloading the original bands and recalculating the indices every time, this function reads the pre-computed index TIFF files from the output directory

def read_index_tif(label, index_name):
    path = f"{OUT_DIR}/{index_name}/{label}_{index_name}.tif"
    with rasterio.open(path) as src:
        return src.read(1).astype(np.float32)

In [ ]:
# Calculating the normalized difference between two bands

def safe_ratio(a, b):
    a = a.astype(np.float32)
    b = b.astype(np.float32)
    with np.errstate(invalid="ignore", divide="ignore"):
        denom  = a + b
        result = np.where(np.abs(denom) > 1e-9, (a - b) / denom, np.nan)
    return result.astype(np.float32)

### Spectral Maths and Features for Model Training

In [ ]:
#Taking the stacked layers and performs matrixoperations to extract mathematical cheatsheets of the spectral indices, which are used to highlight specific features in the satellite images, such as vegetation, water, built-up areas, and burnt land. The function computes NDVI, NDWI, NDBI, NBR, and EVI based on the respective formulas

def compute_indices(stack):
    blue = stack[1];  green = stack[2];  red   = stack[3]
    nir = stack[4];  swir1 = stack[5];  swir2 = stack[6]

    ndvi = np.clip(safe_ratio(nir,   red),   -1.0, 1.0)
    ndwi = np.clip(safe_ratio(green, nir),   -1.0, 1.0)
    ndbi = np.clip(safe_ratio(swir1, nir),   -1.0, 1.0)
    nbr  = np.clip(safe_ratio(nir,   swir2), -1.0, 1.0)

    with np.errstate(invalid="ignore", divide="ignore"):
        d_evi = (nir.astype(np.float32)+ 6.0  * red.astype(np.float32)- 7.5  * blue.astype(np.float32)+ 1.0)
        evi = np.where(
            np.abs(d_evi) > 1e-9,2.5 * (nir.astype(np.float32) - red.astype(np.float32)) / d_evi,np.nan
        )
    evi = np.clip(evi, -1.0, 1.0).astype(np.float32)

    return {"ndvi": ndvi, "ndwi": ndwi, "ndbi": ndbi, "nbr": nbr, "evi": evi}

In [ ]:
def pseudo_labels(ndvi_f, ndwi_f, ndbi_f):
    lab = np.full(len(ndvi_f), -1, dtype=np.int8)
    lab[(ndvi_f > 0.05)  & (ndvi_f < 0.22) & (np.abs(ndbi_f) < 0.12)] = 2
    lab[(ndbi_f > 0.08)  & (ndvi_f < 0.12)] = 3
    lab[(ndwi_f > 0.15)  & (ndvi_f < 0.02)] = 0
    lab[(ndvi_f > 0.32)  & (ndwi_f < -0.05)] = 1
    return lab

### Automated Pseudo Labeling

In [ ]:
#Automatically builing a basic traning dataset without manual labelling, by using strict rules based criteria on the spectral indices to assign pseudo-labels to the pixels, which can then be used to train a machine learning model for land cover classification.
#As an example if water index is very high like more than 0.15 and plt index is very low like less than 0.02 it assign a class label of 0 which is water

def build_feature_matrix(stack, ndvi, ndwi, ndbi, nbr, evi):
    layers = [
        stack[1], stack[2], stack[3],  # B2 B3 B4
        stack[4], stack[5], stack[6],  # B5 B6 B7
        ndvi, ndwi, ndbi, nbr, evi,
    ]
    cube = np.stack(layers, axis=0) # (11, H, W)
    valid_mask = ~np.any(np.isnan(cube), axis=0) # (H, W)  True = usable
    X = cube[:, valid_mask].T # (N_valid, 11)
    return X, valid_mask

### Enviromental Change Metrics

In [ ]:
#Compares vegetation maps form different datas and line up the changes in vegetation cover between two NDVI images, and if a land mask is provided, it excludes ocean and lagoon pixels from the analysis. calculates the number of vegetated pixels in each image, the number of pixels that lost or gained vegetation, and converts these pixel counts into hectares. It also computes the percentage loss, gain, and net change in vegetation cover relative to the first image.

def veg_change_stats(ndvi_t1, ndvi_t2, land_mask=None, thresh=VEG_THRESHOLD):
    h = min(ndvi_t1.shape[0], ndvi_t2.shape[0])
    w = min(ndvi_t1.shape[1], ndvi_t2.shape[1])
    t1, t2 = ndvi_t1[:h, :w], ndvi_t2[:h, :w]
    both_valid = (~np.isnan(t1)) & (~np.isnan(t2))
    if land_mask is not None:
        both_valid = both_valid & land_mask[:h, :w]
    v1 = (t1 > thresh) & both_valid
    v2 = (t2 > thresh) & both_valid
    n1, n2 = int(v1.sum()), int(v2.sum())
    lost   = int((v1 & ~v2).sum())
    gained = int((~v1 & v2).sum())
    return {
        "n_veg_t1": n1, "n_veg_t2": n2,
        "lost_px": lost, "gained_px": gained,
        "lost_ha": round(lost * PX_HA, 1),
        "gained_ha": round(gained * PX_HA, 1),
        "net_ha": round((n2 - n1) * PX_HA, 1),
        "pct_loss": round(100 * lost / n1 if n1 > 0 else 0.0, 2),
        "pct_gain": round(100 * gained / n1 if n1 > 0 else 0.0, 2),
        "pct_net": round(100 * (n2 - n1) / n1 if n1 > 0 else 0.0, 2),
    }

# Rechecking the Data Loaded or Not

In [ ]:
SCENES = discover_scenes(RAW_DIR)
LABELS = sorted(SCENES.keys())

if not LABELS:
    raise RuntimeError(
        f"No valid Landsat images found in {RAW_DIR!r}.\n"
    )

print(f"Found {len(LABELS)} images: {LABELS[0]} to {LABELS[-1]}\n")
for lbl in LABELS:
    meta = SCENES[lbl].get("_meta", {})
    sr = sum(1 for k in SCENES[lbl] if k.startswith("SR_B"))
    qa = "yes" if "QA_PIXEL" in SCENES[lbl] else "NO"
    date = meta["date"].strftime("%Y-%m-%d") if "date" in meta else "?"
    print(f"  {lbl}  {meta.get('sensor','?')}  {date}  SR_bands={sr}  QA={qa}")


### Visualizing the Timeline of Data Collection

In [ ]:
rows = []
for lbl in LABELS:
    meta = SCENES[lbl].get("_meta", {})
    rows.append({
        "Label":    lbl,
        "Sensor":   meta.get("sensor", "?"),
        "Date":     meta["date"].strftime("%Y-%m-%d") if "date" in meta else "?",
        "SR Bands": sum(1 for k in SCENES[lbl] if k.startswith("SR_B")),
        "Has QA":   "yes" if "QA_PIXEL" in SCENES[lbl] else "NO",
    })
df_inv = pd.DataFrame(rows)
print(df_inv.to_string(index=False))



In [ ]:
fig, ax = plt.subplots(figsize=(13, 2.2))
sensor_colors = {"LC08": "Teal", "LC09": "Crimson"}

for _, r in df_inv.iterrows():
    c = sensor_colors.get(r["Sensor"], "gray")
    try:
        dt_num = mdates.date2num(datetime.strptime(r["Date"], "%Y-%m-%d"))
        ax.axvline(dt_num, color=c, lw=5, alpha=0.8)
    except Exception:
        pass

ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.xaxis.set_major_locator(mdates.YearLocator())

ax.legend(handles=[
    mpatches.Patch(color="Teal", label="Landsat 8"),
    mpatches.Patch(color="Crimson", label="Landsat 9"),
], loc="upper left")

ax.set_title("Image Acquisition Timeline", fontweight="bold")
ax.yaxis.set_visible(False)

try:
    all_dates = [datetime.strptime(r["Date"], "%Y-%m-%d")
                 for _, r in df_inv.iterrows() if r["Date"] != "?"]
    if all_dates:
        ax.set_xlim(mdates.date2num(min(all_dates)) - 30,
                    mdates.date2num(max(all_dates)) + 30)
except Exception:
    pass

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/graphs/timelineDataCollection.png", dpi=130, bbox_inches="tight")
plt.show()

# Data Preprocessing and Stacking

In [ ]:
#Once scene at a time, we load the spectral bands, apply the scale and offset to convert to surface reflectance, apply cloud masking using the QA_PIXEL layer, and save the processed data as a stacked multi-band TIFF file. We also log the percentage of cloud cover for each scene in a side text file. If a scene has already been processed (Ex: the output TIFF exists), we skip it and read the cloud cover from the log instead. This way we can run the script multiple times without reprocessing scenes that are already done.

cloud_log = {}

for label in LABELS:
    out_path = f"{OUT_DIR}/stacked/{label}_stack.tif"
    side_path = f"{OUT_DIR}/stacked/{label}_cloud.txt"

    if os.path.exists(out_path):
        try:
            with open(side_path, "r", encoding="utf-8") as fh:
                cloud_log[label] = float(fh.read().strip())
        except (FileNotFoundError, ValueError):
            cloud_log[label] = 0.0
        print(f"  {label}  Done Already  (cloud={cloud_log[label]:.1f}%)")
        continue

    t0 = time.time()
    scene = SCENES[label]
    bands, profile = [], None

    for i in range(1, 8):
        bk = f"SR_B{i}"
        if bk not in scene:
            print(f"  {label}  WARNING: {bk} Image Missing")
            bands = []
            break
        arr, prof = load_band(scene[bk], downsample=DOWNSAMPLE)
        bands.append(arr)
        del arr
        if profile is None:
            profile = prof

    if not bands:
        continue

    stack = np.stack(bands, axis=0).astype(np.float32)   # (7, H, W)
    del bands
    gc.collect()

    # Cloud mask
    if "QA_PIXEL" in scene:
        bad = load_qa(scene["QA_PIXEL"], downsample=DOWNSAMPLE)
        cloud_pct = 100.0 * bad.sum() / bad.size
        stack[:, bad] = np.nan
        del bad
    else:
        cloud_pct = 0.0
        print(f"  {label}  WARNING: no QA Layer")

    cloud_log[label] = round(cloud_pct, 2)

    # Saving the stacked TIF
    p = profile.copy()
    p.update(count=7, dtype="float32", compress="lzw",nodata=float("nan"), BIGTIFF="YES")
    with rasterio.open(out_path, "w", **p) as dst:
        dst.write(stack)
    del stack
    gc.collect()

    with open(side_path, "w", encoding="utf-8") as fh:
        fh.write(str(cloud_log[label]))

    print(f"  {label}  done  ({time.time()-t0:.0f}s  cloud={cloud_pct:.1f}%)")



In [ ]:
fig, ax = plt.subplots(figsize=(13, 3.5))
labels_plot = list(cloud_log.keys())
values_plot = list(cloud_log.values())
bar_colors  = ["green" if v < 30 else "orange" if v < 60 else "red" for v in values_plot]

ax.bar(labels_plot, values_plot, color=bar_colors, edgecolor="white", width=0.65)
ax.axhline(30, color="#666", ls="--", lw=1.2, label="30% threshold")
ax.set_ylabel("Cloud cover (%)")
ax.set_title("Cloud Cover Analysis  (Green<30%  Orange 30-60%  Red>60%)",fontweight="bold")
ax.legend()
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/graphs/cloudCoverAnalysis.png", dpi=130, bbox_inches="tight")
plt.show()

In [ ]:
ncols = 3
nrows = (len(LABELS) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4.5, nrows * 4.2))
axes = np.array(axes).ravel()

for i, label in enumerate(LABELS):
    stk_path = f"{OUT_DIR}/stacked/{label}_stack.tif"
    if not os.path.exists(stk_path):
        axes[i].set_title(f"{label} — missing"); axes[i].axis("off"); continue

    with rasterio.open(stk_path) as src:
        r = src.read(4).astype(np.float32)
        g = src.read(3).astype(np.float32)
        b = src.read(2).astype(np.float32)

    rgb = np.stack([pct_stretch(r), pct_stretch(g), pct_stretch(b)], axis=-1)
    del r, g, b; gc.collect()
    axes[i].imshow(rgb)
    axes[i].set_title(f"True colour — {label}", fontsize=9, fontweight="bold")
    axes[i].axis("off")
    del rgb; gc.collect()

for j in range(len(LABELS), len(axes)):
    axes[j].set_visible(False)

plt.suptitle("True Colour Composites of Images (cloud-masked)", fontsize=13,fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/graphs/trueColorImagesGrid.png", dpi=100, bbox_inches="tight")
plt.show()


# Feature Engineering & Time Series Data Collection

In [ ]:
# Extracting muli layerd TIFF files, computing the spectral indices, and saving each index as a separate single band TIFF file. calculates mathematical statistics on the NDVI index, such as mean, standard deviation, median, percentiles, and percentage of pixels classified as dense vegetation, moderate vegetation, and non-vegetation. These statistics are collected into a time series DataFrame for further analysis and visualization in later steps.

stats_rows = []

for label in LABELS:
    stk_path = f"{OUT_DIR}/stacked/{label}_stack.tif"
    if not os.path.exists(stk_path):
        print(f"  {label}: stacked TIF missing — run Pass 1 first"); continue

    t0 = time.time()
    with rasterio.open(stk_path) as src:
        stack   = src.read().astype(np.float32)
        profile = src.profile.copy()

    idx = compute_indices(stack)
    del stack; gc.collect()

    one_band = profile.copy()
    one_band.update(count=1, dtype="float32", compress="lzw",
                    nodata=float("nan"), BIGTIFF="YES")

    for iname, arr in idx.items():
        out_path = f"{OUT_DIR}/{iname}/{label}_{iname}.tif"
        if not os.path.exists(out_path):
            save_raster(arr, one_band, out_path)

    valid = idx["ndvi"][~np.isnan(idx["ndvi"])].ravel()
    meta  = SCENES[label].get("_meta", {})
    date  = meta.get("date", None)
    if len(valid) > 1000 and date is not None:
        stats_rows.append({
            "label": label,
            "date": date,
            "date_frac": date.year + (date.timetuple().tm_yday - 1) / 365.0,
            "mean_ndvi": float(np.nanmean(idx["ndvi"])),
            "std_ndvi": float(np.nanstd(idx["ndvi"])),
            "median_ndvi": float(np.nanmedian(idx["ndvi"])),
            "p10": float(np.nanpercentile(idx["ndvi"], 10)),
            "p90": float(np.nanpercentile(idx["ndvi"], 90)),
            "pct_dense": float(100 * (idx["ndvi"] > 0.50).sum() / np.isfinite(idx["ndvi"]).sum()),
            "pct_moderate":float(100 * ((idx["ndvi"] >= 0.25) & (idx["ndvi"] <= 0.50)).sum()/ np.isfinite(idx["ndvi"]).sum()),
            "pct_nonveg": float(100 * (idx["ndvi"] <= 0).sum() / np.isfinite(idx["ndvi"]).sum()),
            "n_valid_px": int(np.isfinite(idx["ndvi"]).sum()),
        })

    del idx; gc.collect()
    print(f"  {label}  done  ({time.time()-t0:.0f}s)")

df_ts = pd.DataFrame(stats_rows).sort_values("date").reset_index(drop=True)
print(f"\nTime-series rows: {len(df_ts)}")



In [ ]:
# Creating a presistent land mask that identifies pixels that are likely to be permanent water bodies, based on the NDWI index across all scenes. A pixel is classified as 'permanent water' if it has an NDWI value greater than 0.15 in more than 60% of the scenes where it is valid. This land mask is saved to disk as a TIFF file.

land_mask_path = f"{OUT_DIR}/stats/land_mask.tif"

if os.path.exists(land_mask_path):
    with rasterio.open(land_mask_path) as src:
        LAND_MASK  = src.read(1).astype(bool)
        mask_prof  = src.profile.copy()
    print(f"Land mask loaded from disk  ({LAND_MASK.sum():,} land pixels)")
else:
    print("Building persistent land mask from all NDWI TIFs\n")
    ndwi_labels = sorted(
        l for l in LABELS if os.path.exists(f"{OUT_DIR}/ndwi/{l}_ndwi.tif")
    )
    water_count = None
    total_valid  = None
    ref_profile  = None

    for label in ndwi_labels:
        with rasterio.open(f"{OUT_DIR}/ndwi/{label}_ndwi.tif") as src:
            ndwi_arr = src.read(1).astype(np.float32)
            if ref_profile is None:
                ref_profile = src.profile.copy()

        valid = ~np.isnan(ndwi_arr)
        is_water = (ndwi_arr > 0.15) & valid

        if water_count is None:
            water_count = is_water.astype(np.uint16)
            total_valid  = valid.astype(np.uint16)
        else:
            h = min(water_count.shape[0], is_water.shape[0])
            w = min(water_count.shape[1], is_water.shape[1])
            water_count[:h, :w] += is_water[:h, :w].astype(np.uint16)
            total_valid[:h,  :w] += valid[:h, :w].astype(np.uint16)

        del ndwi_arr, valid, is_water
        gc.collect()
        print(f"  {label} processed")

    if water_count is None:
        raise RuntimeError("No NDWI TIFs found to build land mask")

    with np.errstate(invalid="ignore", divide="ignore"):
        water_frac = np.where(total_valid > 0,water_count / total_valid.astype(np.float32),0.0)
    LAND_MASK = water_frac <= 0.60   # True = land

    # Saving to the disk
    mp = ref_profile.copy()
    mp.update(count=1, dtype="uint8", compress="lzw", nodata=255)
    os.makedirs(os.path.dirname(land_mask_path), exist_ok=True)
    with rasterio.open(land_mask_path, "w", **mp) as dst:
        dst.write(LAND_MASK.astype(np.uint8)[np.newaxis, :, :])

    del water_count, total_valid, water_frac
    gc.collect()
    print(f"\nLand mask saved. Land pixels: {LAND_MASK.sum():,}")



In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
ax.imshow(LAND_MASK, cmap="Greens", interpolation="nearest")
ax.set_title("Persistent Land Mask  (Green = land, White = permanent water)",
             fontweight="bold")
ax.axis("off")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/graphs/presistentLandMark.png", dpi=110, bbox_inches="tight")
plt.show()

In [ ]:
#This isolates specifis region of intrest withing the land mask out of massive landsat satellite image so statistics arent skewd


# Open the pre-calculated land mask to copy its coordinate reference system (CRS),positioning math (transform), and grid dimensions (height and width).
with rasterio.open(f"{OUT_DIR}/stats/land_mask.tif") as src:
    lm_profile = src.profile.copy()
    lm_transform = src.transform
    lm_crs = src.crs
    H_lm, W_lm = src.height, src.width

# Specify the target area boundary using standard WGS84 Latitude and Longitude. The boundary is padded slightly to safely capture the full peninsula and dunes.
lat_min, lat_max = 8.00,  8.65
lon_min, lon_max = 79.68, 79.90

# Convert WGS84 → raster CRS
try:
    # Set up a coordinate transformer to convert GPS degrees (EPSG:4326) into the specific metric projected coordinate system used by the satellite image.
    wgs84_to_utm = Transformer.from_crs("EPSG:4326", lm_crs, always_xy=True)
    x_min, y_min = wgs84_to_utm.transform(lon_min, lat_min)
    x_max, y_max = wgs84_to_utm.transform(lon_max, lat_max)

    # Translate those metric ground coordinates into raw matrix pixel addresses (row/column numbers).
    from rasterio.transform import rowcol
    row_max, col_min = rowcol(lm_transform, x_min, y_min)
    row_min, col_max = rowcol(lm_transform, x_max, y_max)

    # Clamp coordinates to ensure the bounding box values fall within valid image arrays and don't cause an "out of bounds" index error.
    row_min = max(0, int(row_min))
    row_max = min(H_lm - 1, int(row_max))
    col_min = max(0, int(col_min))
    col_max = min(W_lm - 1, int(col_max))

    # Build ROI mask (True = Kalpitiya peninsula)
    ROI_MASK = np.zeros((H_lm, W_lm), dtype=bool) # Generate an empty, blank boolean array (all False) matching the map's full size.
    ROI_MASK[row_min:row_max+1, col_min:col_max+1] = True # Paint the calculated bounding box area as True (White) inside the empty array.

    # Combine masks using Bitwise AND: A pixel must be inside the bounding box AND it must be actual land (not lagoon or sea) to equal True.
    KALP_MASK = LAND_MASK & ROI_MASK
    # Count the remaining true land pixels inside the peninsula crop.
    kalp_px = int(KALP_MASK.sum())
    kalp_ha = kalp_px * PX_HA # Convert pixel counts to real-world surface area in hectares.

    print(f"Kalpitiya ROI pixel range : rows {row_min}–{row_max},  cols {col_min}–{col_max}")
    print(f"Kalpitiya land pixels : {kalp_px:,}")
    print(f"Kalpitiya land area : {kalp_ha:,.0f} ha  (at {30*DOWNSAMPLE}m resolution)")
    print()
    print("Compare with full tile:")
    print(f"  Full tile land pixels : {LAND_MASK.sum():,}  ({LAND_MASK.sum()*PX_HA:,.0f} ha)")
    print(f"  Kalpitiya fraction : {100*kalp_px/LAND_MASK.sum():.1f}% of tile land")

    # Saving the Kalpitiya mask
    with rasterio.open(f"{OUT_DIR}/stats/kalpitiya_mask.tif", "w",
                        driver="GTiff", height=H_lm, width=W_lm,
                        count=1, dtype="uint8", crs=lm_crs,
                        transform=lm_transform, compress="lzw") as dst:
        dst.write(KALP_MASK.astype(np.uint8)[np.newaxis])
    print(f"\nKalpitiya mask saved: {OUT_DIR}/stats/kalpitiya_mask.tif")

except Exception as e:
    print(f"Could not build Kalpitiya ROI: {e}, Check that pyproj is installed and the coordinate transformation is working.")
    print("Falling back to full land mask (LAND_MASK).")
    KALP_MASK = LAND_MASK


In [ ]:
ncols = 4
nrows = (len(LABELS) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4.3, nrows * 4.0))
axes = np.array(axes).ravel()

for i, label in enumerate(LABELS):
    ndvi_path = f"{OUT_DIR}/ndvi/{label}_ndvi.tif"
    if not os.path.exists(ndvi_path):
        axes[i].set_title(f"{label} — Image Missing"); axes[i].axis("off"); continue

    with rasterio.open(ndvi_path) as src:
        arr = src.read(1).astype(np.float32)

    im = axes[i].imshow(arr, cmap="RdYlGn", vmin=-0.15, vmax=0.75)
    axes[i].set_title(f"NDVI  {label}", fontsize=9, fontweight="bold")
    axes[i].axis("off")
    plt.colorbar(im, ax=axes[i], fraction=0.046, pad=0.03, shrink=0.85)
    del arr; gc.collect()

# Deep Reds / Yellows (-0.15 to 0.10): Non-vegetated surfaces
#   Examples: Open water, bare soil, sand dunes, asphalt, or concrete buildings.

# Light Greens (0.20 to 0.40): Sparse / Low-density vegetation
#   Examples: Seasonal grass, scrublands, open savannas, or young crops.

# Deep, Vibrant Greens (>= 0.60): Dense / Healthy plant canopies
#   Examples: Thick forests, mangrove stands, or thriving agricultural plots.

for j in range(len(LABELS), len(axes)):
    axes[j].set_visible(False)

plt.suptitle("NDVI of all Images", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/graphs/ndviImagesGrid.png", dpi=100, bbox_inches="tight")
plt.show()



In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
cmap_fn = matplotlib.colormaps.get_cmap("viridis")
palette  = [cmap_fn(x) for x in np.linspace(0.1, 0.9, len(LABELS))]

for i, label in enumerate(LABELS):
    ndvi_path = f"{OUT_DIR}/ndvi/{label}_ndvi.tif"
    if not os.path.exists(ndvi_path):
        continue
    with rasterio.open(ndvi_path) as src:
        arr = src.read(1).astype(np.float32)
    valid = arr[~np.isnan(arr)].ravel()
    del arr; gc.collect()
    if len(valid) > 300_000:
        rng   = np.random.default_rng(42)
        valid = rng.choice(valid, 300_000, replace=False)
    ax.hist(valid, bins=150, density=True, histtype="stepfilled",
            alpha=0.45, color=palette[i], edgecolor="none",
            label=label.replace("_", "/"))
    del valid

ax.axvline(VEG_THRESHOLD, color="black", ls="--", lw=1.5,
           label=f"Veg threshold ({VEG_THRESHOLD})")
ax.set_xlabel("NDVI")
ax.set_ylabel("Density")
ax.set_title("NDVI distributions across all scenes",fontweight="bold")
ax.legend(ncol=3, fontsize=8)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/graphs/ndviHistGraph.png", dpi=130, bbox_inches="tight")
plt.show()

